# Train / Validation / Test Split

Creates canonical, reusable splits for all experiment notebooks.

## Strategy

**Leave-One-Tournament-Out (LOTO) cross-validation** over the 4 FIFA World Cups (2010, 2014, 2018, 2022).

| Fold | Train on | Evaluate on | n eval |
|------|----------|-------------|--------|
| WC 2010 | all matches before 2010-06-01 | WC 2010 (64 matches) | 64 |
| WC 2014 | all matches before 2014-06-01 | WC 2014 (64 matches) | 64 |
| WC 2018 | all matches before 2018-06-01 | WC 2018 (64 matches) | 64 |
| WC 2022 | all matches before 2022-11-01 | WC 2022 (64 matches) | 64 |

Pooling all 4 folds gives **256 evaluation matches** (vs 64–192 before), shrinking the 95% CI from ~±0.68 to ~±0.37 pts/match.

Temporal ordering is strict: each fold's training data ends before that WC starts — no leakage.

### Final test
WC 2022 fold also serves as the **held-out final test** (most recent, closest era to WC 2026). Report LOTO-CV mean as the primary comparison metric; WC 2022 alone as the secondary.

### Autofill baseline
Measured empirically: predict **1-0 to the ELO-favourite** (1-1 if |elo_diff| < 50). This mimics the FIFA-ranking autofill strategy Sporza uses as default.

In [1]:
import sys
sys.path.insert(0, '../src')

import hashlib
import json
import numpy as np
import pandas as pd
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA_INTERIM  = Path('../data/interim')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

WC_TOURNAMENT = 'FIFA World Cup'
WC_YEARS = [2010, 2014, 2018, 2022]

# Tournament start dates (cutoff for training data in each fold)
WC_CUTOFFS = {
    2010: pd.Timestamp('2010-06-01'),
    2014: pd.Timestamp('2014-06-01'),
    2018: pd.Timestamp('2018-06-01'),
    2022: pd.Timestamp('2022-11-01'),
}

In [2]:
df = pd.read_parquet(DATA_INTERIM / 'train_features.parquet')
print(f'Full dataset: {len(df):,} rows, {df["date"].min().date()} – {df["date"].max().date()}')

Full dataset: 14,681 rows, 2010-01-02 – 2026-06-11


In [3]:
# Build LOTO folds
folds = {}
for year in WC_YEARS:
    cutoff = WC_CUTOFFS[year]
    wc_mask = (df['tournament'] == WC_TOURNAMENT) & (df['date'].dt.year == year)
    train_fold = df[df['date'] < cutoff].copy()
    eval_fold  = df[wc_mask].copy()
    assert len(eval_fold) == 64, f'WC {year} has {len(eval_fold)} matches (expected 64)'
    # No leakage
    overlap = pd.merge(train_fold[['date','home_team','away_team']],
                       eval_fold[['date','home_team','away_team']],
                       on=['date','home_team','away_team'])
    assert len(overlap) == 0, f'Leakage in fold {year}'
    folds[year] = {'train': train_fold, 'eval': eval_fold}
    print(f'WC {year}: train={len(train_fold):,} rows (up to {cutoff.date()})  eval=64 matches')

# Pool all eval splits
eval_all = pd.concat([folds[y]['eval'] for y in WC_YEARS], ignore_index=True)
print(f'\nPooled eval: {len(eval_all)} matches across {len(WC_YEARS)} WC tournaments')

WC 2010: train=219 rows (up to 2010-06-01)  eval=64 matches
WC 2014: train=3,900 rows (up to 2014-06-01)  eval=64 matches
WC 2018: train=7,272 rows (up to 2018-06-01)  eval=64 matches
WC 2022: train=11,106 rows (up to 2022-11-01)  eval=64 matches

Pooled eval: 256 matches across 4 WC tournaments


In [4]:
# Score distributions per fold
print('Score distributions:')
for year in WC_YEARS:
    s = folds[year]['eval']
    n = len(s)
    print(f'  WC {year}: H-win {(s.home_score > s.away_score).mean():.1%}  '
          f'Draw {(s.home_score == s.away_score).mean():.1%}  '
          f'A-win {(s.home_score < s.away_score).mean():.1%}  '
          f'avg {s.home_score.mean():.2f}–{s.away_score.mean():.2f}')
print(f'  Pooled:  H-win {(eval_all.home_score > eval_all.away_score).mean():.1%}  '
      f'Draw {(eval_all.home_score == eval_all.away_score).mean():.1%}  '
      f'A-win {(eval_all.home_score < eval_all.away_score).mean():.1%}  '
      f'avg {eval_all.home_score.mean():.2f}–{eval_all.away_score.mean():.2f}')

Score distributions:
  WC 2010: H-win 37.5%  Draw 25.0%  A-win 37.5%  avg 1.20–1.06
  WC 2014: H-win 45.3%  Draw 20.3%  A-win 34.4%  avg 1.31–1.36
  WC 2018: H-win 39.1%  Draw 20.3%  A-win 40.6%  avg 1.38–1.27
  WC 2022: H-win 43.8%  Draw 23.4%  A-win 32.8%  avg 1.55–1.14
  Pooled:  H-win 41.4%  Draw 22.3%  A-win 36.3%  avg 1.36–1.21


In [5]:
# Empirical autofill baseline: predict 1-0 to ELO-favourite; 1-1 if near-equal
ELO_THRESHOLD = 50  # treat as roughly equal if |elo_diff| < 50

def autofill_predict(split: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """ELO-favourite wins 1-0; near-equal ELO → predict 1-1 (draw)."""
    pred_home = pd.Series(np.where(split['elo_diff'].values >= 0, 1, 0), index=split.index)
    pred_away = pd.Series(np.where(split['elo_diff'].values < 0, 1, 0), index=split.index)
    # Near-equal: predict 1-1
    near_equal = split['elo_diff'].abs() < ELO_THRESHOLD
    pred_home[near_equal] = 1
    pred_away[near_equal] = 1
    return pred_home, pred_away

print('Autofill baseline (1-0 favourite / 1-1 near-equal):')
all_ph, all_pa = [], []
for year in WC_YEARS:
    s = folds[year]['eval']
    ph, pa = autofill_predict(s)
    bd = score_breakdown(ph, pa, s['home_score'], s['away_score'])
    all_ph.append(ph)
    all_pa.append(pa)
    print(f'  WC {year}: {bd["mean_pts"]:.3f} pts/match  exact={bd["pct_exact"]:.1%}  correct_result={bd["pct_correct_result"]:.1%}')

ph_pool = pd.concat(all_ph, ignore_index=True)
pa_pool = pd.concat(all_pa, ignore_index=True)
bd_pool = score_breakdown(ph_pool, pa_pool, eval_all['home_score'].reset_index(drop=True), eval_all['away_score'].reset_index(drop=True))
print(f'  Pooled:  {bd_pool["mean_pts"]:.3f} pts/match  exact={bd_pool["pct_exact"]:.1%}  correct_result={bd_pool["pct_correct_result"]:.1%}')

Autofill baseline (1-0 favourite / 1-1 near-equal):
  WC 2010: 4.281 pts/match  exact=18.8%  correct_result=51.6%
  WC 2014: 4.016 pts/match  exact=10.9%  correct_result=51.6%
  WC 2018: 4.500 pts/match  exact=18.8%  correct_result=56.2%
  WC 2022: 3.750 pts/match  exact=6.2%  correct_result=54.7%
  Pooled:  4.137 pts/match  exact=13.7%  correct_result=53.5%


In [6]:
# Bootstrap CI function — reusable across all experiment notebooks
def bootstrap_ci(pts_series: pd.Series, n_boot: int = 10000, ci: float = 0.95) -> dict:
    """Bootstrap confidence interval for mean Sporza pts/match."""
    rng = np.random.default_rng(42)
    vals = pts_series.values
    boot_means = np.array([
        rng.choice(vals, size=len(vals), replace=True).mean()
        for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(boot_means, [alpha * 100, (1 - alpha) * 100])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi), 'ci_width': float(hi - lo)}

# CI on pooled autofill
pts_autofill = sporza_points_series(ph_pool, pa_pool,
                                    eval_all['home_score'].reset_index(drop=True),
                                    eval_all['away_score'].reset_index(drop=True))
ci_autofill = bootstrap_ci(pts_autofill)
print(f'Autofill pooled: {ci_autofill["mean"]:.3f} pts/match  '
      f'95% CI [{ci_autofill["ci_lo"]:.3f}, {ci_autofill["ci_hi"]:.3f}]  '
      f'(width {ci_autofill["ci_width"]:.3f})')

Autofill pooled: 4.137 pts/match  95% CI [3.742, 4.539]  (width 0.797)


In [7]:
# Save folds and pooled eval for reuse
for year in WC_YEARS:
    folds[year]['train'].to_parquet(DATA_PROCESSED / f'train_fold_{year}.parquet', index=False)
    folds[year]['eval'].to_parquet(DATA_PROCESSED / f'eval_fold_{year}.parquet', index=False)

eval_all.to_parquet(DATA_PROCESSED / 'eval_pooled.parquet', index=False)

# Also keep a single 'train' for non-LOTO notebooks (train on everything before WC 2022)
train_full = df[df['date'] < WC_CUTOFFS[2022]].copy()
train_full.to_parquet(DATA_PROCESSED / 'train.parquet', index=False)

def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()[:8]

manifest = {
    'source': str(DATA_INTERIM / 'train_features.parquet'),
    'strategy': 'leave-one-tournament-out CV over WC 2010/2014/2018/2022',
    'wc_cutoffs': {str(y): str(c.date()) for y, c in WC_CUTOFFS.items()},
    'folds': {
        str(year): {
            'train_rows': len(folds[year]['train']),
            'eval_rows': 64,
            'train_md5': file_hash(DATA_PROCESSED / f'train_fold_{year}.parquet'),
            'eval_md5':  file_hash(DATA_PROCESSED / f'eval_fold_{year}.parquet'),
        } for year in WC_YEARS
    },
    'eval_pooled': {'rows': len(eval_all), 'md5': file_hash(DATA_PROCESSED / 'eval_pooled.parquet')},
    'autofill_baseline': {
        'rule': '1-0 to ELO-favourite; 1-1 if |elo_diff|<50',
        'pooled_mean_pts': ci_autofill['mean'],
        'ci_95': [ci_autofill['ci_lo'], ci_autofill['ci_hi']],
    },
}
(DATA_PROCESSED / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "source": "../data/interim/train_features.parquet",
  "strategy": "leave-one-tournament-out CV over WC 2010/2014/2018/2022",
  "wc_cutoffs": {
    "2010": "2010-06-01",
    "2014": "2014-06-01",
    "2018": "2018-06-01",
    "2022": "2022-11-01"
  },
  "folds": {
    "2010": {
      "train_rows": 219,
      "eval_rows": 64,
      "train_md5": "c2a72560",
      "eval_md5": "29cf4eeb"
    },
    "2014": {
      "train_rows": 3900,
      "eval_rows": 64,
      "train_md5": "ed25225c",
      "eval_md5": "1613e902"
    },
    "2018": {
      "train_rows": 7272,
      "eval_rows": 64,
      "train_md5": "ec120bf6",
      "eval_md5": "08c32ae5"
    },
    "2022": {
      "train_rows": 11106,
      "eval_rows": 64,
      "train_md5": "dafe396c",
      "eval_md5": "f3375c42"
    }
  },
  "eval_pooled": {
    "rows": 256,
    "md5": "6d42c00a"
  },
  "autofill_baseline": {
    "rule": "1-0 to ELO-favourite; 1-1 if |elo_diff|<50",
    "pooled_mean_pts": 4.13671875,
    "ci_95": [
      3.7421